## Schrodinger equation

In [ ]:
from solver import *
from psiop import *

t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Simple advection equation: u_t = -u_x  (symbol = i*xi)
# Note: psiOp(i*xi) corresponds to -d/dx
eq = Eq(diff(u, t), psiOp(-1j * xi, u)) 
solver = PDESolver(eq)

def initial_condition(x):
    return np.exp(-((x + 2)**2) / 0.5)  # Gaussian offset from center

solver = PDESolver(eq)
solver.setup(
    Lx=10.0, Nx=256, Lt=10.0, Nt=1000,
    boundary_condition='periodic',  # Sponge works with periodic BCs
    initial_condition=initial_condition,
    sponge=True,
    sponge_width=0.2,      # 20% of domain on each side
    sponge_strength=5.0,   # Moderate damping
    sponge_order=2,
)
solver.solve()

# Check if amplitude decreased
initial_amp = np.max(np.abs(solver.frames[0]))
final_amp = np.max(np.abs(solver.frames[-1]))
print(f"Initial amplitude: {initial_amp:.4f}")
print(f"Final amplitude: {final_amp:.4f}")
print(f"Absorption: {100*(1-final_amp/initial_amp):.1f}%")

plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # 'contour', 'front' or None
    mode='surface', # 'imshow' or 'surface'
    physical=True # True or False
)

HTML(ani.to_jshtml())

## Wave equation

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Wave equation
eq = Eq(diff(u, t, t), psiOp(-xi**2, u))
solver = PDESolver(eq)

def initial_condition(x):
    return np.exp(-(x / 0.3)**2)

def initial_velocity(x):
    return 0.0 * x

solver.setup(
    Lx=10.0, Nx=256,
    Lt=10.0, Nt=500,
    boundary_condition='periodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
    # ── Sponge layer ──
    sponge=True,
    sponge_width=0.1,  
    sponge_strength=5,  # max damping rate
    sponge_order=3,        # polynomial ramp order
)

frames = solver.solve()

# Check if amplitude decreased
initial_amp = np.max(np.abs(solver.frames[0]))
final_amp = np.max(np.abs(solver.frames[-1]))
print(f"Initial amplitude: {initial_amp:.4f}")
print(f"Final amplitude: {final_amp:.4f}")
print(f"Absorption: {100*(1-final_amp/initial_amp):.1f}%")

plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # 'contour', 'front' or None
    mode='surface', # 'imshow' or 'surface'
    physical=True # True or False
)

HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from sympy import symbols, Function, Eq, diff

# Import from your local modules
from solver import PDESolver, psiOp

# ─────────────────────────────────────────────────────────
# 1. Define Symbols and the 2D Wave Equation
# ─────────────────────────────────────────────────────────
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u = Function('u')(t, x, y)

# u_tt = Delta u  <==>  u_tt = psiOp(-(xi^2 + eta^2), u)
# Using psiOp forces the solver through the pseudo-differential 
# time-stepping path, which correctly applies the sponge layer.
eq = Eq(diff(u, t, t), psiOp(-(xi**2 + eta**2), u))

solver = PDESolver(eq)

# ─────────────────────────────────────────────────────────
# 2. Define Initial Conditions (Localized 2D Gaussian)
# ─────────────────────────────────────────────────────────
def initial_condition(x, y):
    # A tight Gaussian bump in the center of the domain
    return np.exp(-(x**2 + y**2) / 0.5)

def initial_velocity(x, y):
    # Released from rest
    return 0.0 * x  

# ─────────────────────────────────────────────────────────
# 3. Setup the Solver with the Sponge Layer
# ─────────────────────────────────────────────────────────
solver.setup(
    Lx=10.0, Ly=10.0,     # Domain is [-5, 5] x [-5, 5]
    Nx=128, Ny=128,       # Grid resolution
    Lt=10.0, Nt=1000,       # Simulate long enough for the wave to hit the edges
    boundary_condition='periodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
    
    # ── Sponge Layer Configuration ──
    sponge=True,
    sponge_width=0.20,     # Damping starts at 20% from the edges (r = 4.0)
    sponge_strength=30.0,  # Damping rate (high enough to kill the wave)
    sponge_order=3,        # Smooth polynomial ramp to minimize back-scatter
    
    n_frames=100,
    plot=False
)

# ─────────────────────────────────────────────────────────
# 4. Solve and Verify Absorption
# ─────────────────────────────────────────────────────────
frames = solver.solve()

initial_amp = np.max(np.abs(solver.frames[0]))
final_amp = np.max(np.abs(solver.frames[-1]))

print(f"Initial amplitude: {initial_amp:.4f}")
print(f"Final amplitude:   {final_amp:.4f}")
print(f"Absorption:        {100 * (1 - final_amp / initial_amp):.1f}%")

# ─────────────────────────────────────────────────────────
# 5. Animate the Result
# ─────────────────────────────────────────────────────────
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None,       # 'contour', 'front' or None
    mode='surface',     # 'imshow' is faster, 'surface' is 3D
    physical=True       # True keeps the X/Y axes scaled to physical Lx, Ly
)

# Display in Jupyter Notebook
HTML(ani.to_jshtml())